In [ ]:
import sys

from typing import List, Dict, Iterable, Tuple

#Please do not remove package declarations because these are used by the autograder.

# Insert your genome_path function here, along with any subroutines you need
def genome_path(path: List[str]) -> str:
    """Forms the genome path formed by a collection of patterns."""
    k=len(path[0])
    first=path[0]
    for pattern in path[1:]:
        if pattern[:k-1]==first[len(first)-k+1:]:
            first+=pattern[k-1:]

    return first



def de_bruijn_kmers(k_mers: List[str]) -> Dict[str, List[str]]:
    """Forms the de Bruijn graph of a collection of k-mers."""
    output=dict()
    for k_mer in k_mers:
        prefix=k_mer[:-1]
        suffix=k_mer[1:]
        if prefix in output:
            output[prefix].append(suffix)
        else:
            output[prefix]=[suffix]
    return output



# Insert your ContigGeneration function here, along with any subroutines you need
def ContigGeneration(Patterns: List[str]) -> List[str]:
    
    db = de_bruijn_kmers(Patterns)
    
    in_deg = {}
    out_deg = {}
    for u in db:
        out_deg[u] = len(db[u])
        in_deg.setdefault(u, 0)
        for v in db[u]:
            in_deg[v] = in_deg.get(v, 0) + 1
    for u in db:
        for v in db[u]:
            if v not in out_deg:
                out_deg[v] = 0
    
    contigs = []
    visited = set()
    
    
    for node in set(in_deg.keys()) | set(out_deg.keys()):
        i = in_deg.get(node, 0)
        o = out_deg.get(node, 0)
        
        # if something is not a single path (in 1 and out 1)
        if not (i == 1 and o == 1):
            if node in db:
                for idx, next_node in enumerate(db[node]):
                    if (node, next_node, idx) in visited:
                        continue
                    
                    path = [node, next_node]
                    visited.add((node, next_node, idx))
                    current = next_node
                    
                    # Extend while 1-in-1-out
                    while in_deg.get(current, 0) == 1 and out_deg.get(current, 0) == 1:
                        if current not in db:
                            break
                        next_current = db[current][0]
                        if (current, next_current, 0) in visited:
                            break
                        visited.add((current, next_current, 0))
                        path.append(next_current)
                        current = next_current
                    
                    contigs.append(genome_path(path))
    
    # Handle isolated cycles
    for node in db:
        for idx, next_node in enumerate(db[node]):
            if (node, next_node, idx) not in visited:
                path = [node, next_node]
                visited.add((node, next_node, idx))
                current = next_node
                
                while current != node and current in db:
                    next_current = db[current][0]
                    if (current, next_current, 0) in visited:
                        break
                    visited.add((current, next_current, 0))
                    path.append(next_current)
                    current = next_current
                
                contigs.append(genome_path(path))
    
    return sorted(contigs)


In [ ]:
# Contig Generation Problem: Generate the contigs from a collection of reads (with imperfect coverage).

# Input: A collection of k-mers Patterns.
# Output: All contigs in DeBruijn(Patterns).
# Code Challenge: Solve the Contig Generation Problem.

# If you have difficulties finding maximal non-branching paths in a graph, check out Charging Station: Maximal Non-Branching Paths in a Graph.

# Debug Datasets

# Sample Input 1:

# ATG ATG TGT TGG CAT GGA GAT AGA
# Sample Output 1:

# ATG ATG TGT TGGA CAT GAT AGA
# Sample Input 2:

# AG GT GC TA
# Sample Output 2:

# GTAG GC
# Sample Input 3:

# GTT TTA TAC TTT
# Sample Output 3:

# GTT TTAC TTT
# Sample Input 4:

# GAGA AGAG AACG ACGT ACGG
# Sample Output 4:

# AACG ACGT ACGG AGAGA
# Sample Input 5:

# TGAG GACT CTGA ACTG CTGA
# Sample Output 5:

# TGAG GACTG CTGA CTGA